In [ ]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Stage 1: Data Ingestion

In [4]:
from langchain_community.document_loaders import TextLoader  

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16100\570144785.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
loader = TextLoader(r"F:\AI Projects\End-to-End-LLMOPs-Project\data\paul_graham_essay.txt", encoding="utf8")
documents = loader.load()

In [9]:
len(documents)

1

In [10]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'\t\t\n\nWhat I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data proces'

In [12]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:


text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [15]:

text_chunks=text_splitter.split_documents(documents)

In [16]:

len(text_chunks)

487

In [ ]:
! uv pip install faiss-cpu langchain-huggingface sentence-transformers

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(embedding_model)

f:\AI Projects\End-to-End-LLMOPs-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 199.01it/s]


model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [20]:
from langchain_community.vectorstores import FAISS

In [21]:
vectorstore=FAISS.from_documents(text_chunks, embedding_model)

In [22]:
vectorstore

# Stage 2: Data Ingestion

In [23]:

retriever=vectorstore.as_retriever()

In [25]:
# Perform similarity search
query = "What was the first computer Paul Graham programmed on?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
The first of my friends to get a microcomputer built it himself. It was sold as a kit by Heathkit. I remember vividly how impressed and envious I felt watching him sitting in front of it, typing
--------------------------------------------------
Document 2:
was going to be normal desktop software, which in those days meant Windows software. That was an alarming prospect, because neither of us knew how to write Windows software or wanted to learn. We
--------------------------------------------------
Document 3:
was working harder than I'd ever worked on anything. Occasionally after wrestling for hours with some gruesome bug I'd check Twitter or HN and see someone asking "Does Paul Graham still code?"
--------------------------------------------------
Document 4:
With microcomputers, everything changed. Now you could have a computer sitting right in front of you, on a desk, that could respond to your keystrokes as it was running instead of just churning
---------------------

# Stage 3: Data Ingestion

In [28]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [29]:
prompt=ChatPromptTemplate.from_template(template)

In [30]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [32]:
from langchain_core.output_parsers import StrOutputParser

In [33]:
output_parser=StrOutputParser()

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

In [39]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [42]:
rag_chain.invoke("What inspired the creation of Y Combinator?")

'Based on the provided context, the creation of Y Combinator began with the idea of operating as an "angel firm," a concept where those two words did not typically go together at the time. Additionally, the name "Y Combinator" itself was inspired by "one of the coolest tricks in the lambda calculus." The founders originally called the project "Cambridge Seed" but chose to rename it to avoid a regional name in case the model was copied in Silicon Valley.'

In [44]:
rag_chain.invoke("Which programming language did he learn for AI?")

'He learned Lisp for AI. Since there were no AI classes at Cornell at the time, he had to teach himself, which meant learning Lisp because it was regarded as the language of AI in those days.'

In [45]:
query = "Which programming language did he learn for AI?"

response = rag_chain.invoke(query)

print("=" * 60)
print("Question:")
print(query)
print("\nAnswer:")
print(response)
print("=" * 60)

Question:
Which programming language did he learn for AI?

Answer:
He learned Lisp for AI. Since there were no AI classes at Cornell at the time, he had to teach himself, which meant learning Lisp because it was regarded as the language of AI in those days.
